# US3.8 — Robustness / Overfitting Gate

**Frozen validation only.** No result from this notebook may promote or replace the US3.5 six candidates or three controls. Primary model is REAL 1% participation; US3.7 pending-order handling and US3.5 corporate-event corrections are inherited. The 55/45, beta exit, and session-frequency grids below are diagnostic surfaces only.

This notebook is a separate work product. It will not modify US3.5/3.6/3.7 result files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, pandas as pd, os, runpy
ROOT = Path('/content/drive/MyDrive/미국주식데이터')
OUT = ROOT / 'US38_robustness_gate_20260928'
OUT.mkdir(exist_ok=True)
required = ['descriptions.csv','actions.csv','metrics.csv','stocks.csv','tickers.csv','sp500.csv','funds.csv']
missing = [x for x in required if not (ROOT/x).exists()]
assert not missing, f'Missing required universe files: {missing}'
print('Drive mounted; all 7 universe CSVs present')
print({x:(ROOT/x).stat().st_size for x in required})
assert (ROOT/'US35_followup_20260928'/'simulation_complete.json').exists()
assert (ROOT/'US35_followup_20260928'/'summary.csv').exists()
assert (ROOT/'US33_validation_20260927'/'canonical_feature_inputs.parquet').exists()
manifest = json.loads((ROOT/'manifest.json').read_text()) if (ROOT/'manifest.json').exists() else None
print('Source gates: US33 canonical inputs + US35 results present; manifest:', bool(manifest))

In [ ]:
# Gate-first runner pinned to the research commit; no changes to production/main.
%pip -q install duckdb==1.3.2 pandas==2.2.3 pyarrow==23.0.1 scipy
import urllib.request, subprocess, sys, time
assert 'process' not in globals() or process.poll() is not None, 'A replay is already running.'
CODE_COMMIT = '628e1073c63d7b789a12d468638e894c5304035f'
RUN_ID = time.strftime('replay_%Y%m%dT%H%M%SZ', time.gmtime())
CODE = Path('/content/us38_gate_code'); CODE.mkdir(exist_ok=True)
for name in ['run_us38_gate.py','analyze_us38_gate.py']:
    url = f'https://raw.githubusercontent.com/Jiwoon625/cloudtrend/{CODE_COMMIT}/research/us38-robustness-gate/colab/{name}'
    urllib.request.urlretrieve(url, CODE/name)
RUN_OUT = OUT/RUN_ID
RUN_OUT.mkdir(exist_ok=False)
LOG = RUN_OUT/'execution.log'
env = dict(os.environ, US38_RUN_ID=RUN_ID, US38_CODE_COMMIT=CODE_COMMIT, PYTHONUNBUFFERED='1')
with LOG.open('w') as log:
    process = subprocess.Popen([sys.executable, '-u', str(CODE/'run_us38_gate.py')], stdout=log, stderr=subprocess.STDOUT, env=env)
print('STARTED',process.pid,'RESULTS',RUN_OUT)


In [ ]:
# Read progress without starting another calculation.
print('PROCESS_EXIT_CODE', process.poll())
print(LOG.read_text()[-6000:])


In [ ]:
assert (RUN_OUT/'baseline_gate.json').exists(), 'Baseline pending: inspect execution.log.'
gate = json.loads((RUN_OUT/'baseline_gate.json').read_text())
display(pd.DataFrame(gate['checks']))
assert gate['passed'], 'Parity failed: stop robustness interpretation.'


In [ ]:
assert (RUN_OUT/'simulation_complete.json').exists(), 'Replay/diagnostics pending.'
print((RUN_OUT/'simulation_complete.json').read_text())
display(pd.read_csv(RUN_OUT/'all_metrics.csv').query("period == 'full'"))
print(json.loads((RUN_OUT/'pit_source_audit.json').read_text())['gate'])


## Interpretation restrictions
Baseline parity runs first. Six candidates and three controls are frozen. Primary execution is REAL 1% with controlled persistent pending orders. Fixed-session surfaces are distinct from calendar schedules. Fresh starts hold only cash; rolling windows inherit the continuous portfolio. Stock/sector exclusions replay portfolios while preserving ranks; year exclusions remove contribution chains only. Strict PIT remains NOT PASSED / PENDING unless separately reconstructed and replayed. No Production promotion.
